# 11e — Jeux globaux : la coordination avec information incomplète (Kay–Perry)

**Autour de 11** — prolonge [11 · Jeux bayésiens](GameTheory-11-BayesianGames-Python.ipynb), [04 · Équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb) et le jumeau [11c — sélection d'équilibre par trembles](GameTheory-11c-Global-Games-Python.ipynb) (mêmes jeux globaux, autre question : ici le bruit tranche vers **l'équilibre efficace**, là vers l'issue fondamentale).

Un jeu de coordination 2×2 à information **complète** porte **deux** équilibres de Nash purs (les deux accords) plus un équilibre mixte instable : la **multiplicité** n'est pas tranchée. Le dispositif des **jeux globaux** (Kay & Perry 2006 ; Hagen & Köbis 2016) donne à chaque joueur un **signal bruité** d'un fondamental commun `θ`.

**La question de ce carnet** : le bruit **résout-il** la multiplicité, et, s'il le fait, **quel** équilibre **sélectionne-t-il** ? La réponse, mesurée ci-dessous : oui — il ne reste **qu'un seul** équilibre (un seuil unique), et, quand le bruit disparaît (`σ → 0`), il sélectionne **l'équilibre efficace** dès que le fondamental le mérite.

> Cœur du carnet : tout est **mesuré** (intégration gaussienne en forme close + recherche de racine), sans approximation — le résultat est **falsifiable**.

In [1]:
import numpy as np
import math

# --- Loi normale standard, via la stdlib (pas besoin de scipy) ---
def phi(x):
    """Densité normale standard : phi(x) = exp(-x^2/2) / sqrt(2 pi)."""
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def Phi(x):
    """Fonction de répartition normale standard : Phi(x) = 0.5 (1 + erf(x/sqrt 2))."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# --- Paramètres du jeu (partagés par tout le carnet) ---
TAU2 = 1.0   # variance a priori du fondamental theta ~ N(0, TAU2)
C    = 1.0   # coût d'agir seul : payer H face a L rapporte -C

np.random.seed(42)   # reproductibilité des tirages (si simulation)
print("Imports OK  |  phi(0) =", round(phi(0), 4), " |  Phi(0) =", Phi(0), " |  C =", C)

Imports OK  |  phi(0) = 0.3989  |  Phi(0) = 0.5  |  C = 1.0


## 1. Le jeu de coordination complet — la multiplicité d'origine

Deux joueurs, chacun joue **H** (investir / entrer) ou **L** (s'abstenir). Il existe un **fondamental** `θ` (par ex. la rentabilité d'un projet) : jouer H rapporte `θ` **si l'autre joue aussi H**, mais coûte `C` si l'on joue H **seul**.

| Moi \ Adverse | **H** | **L** |
|---|---|---|
| **H** | `θ` | `−C` |
| **L** | `0` | `0` |

À **information complète** (chaque joueur connaît `θ`) :
- **(H,H)** est un équilibre de Nash **si `θ > 0`** (H est une meilleure réponse à H) ;
- **(L,L)** est **toujours** un équilibre (face à L, L bat H : `0 > −C`) ;
- l'**équilibre mixte** (chacun joue H avec probabilité `p* = C/(θ+C)`) existe quand `θ > 0`.

**Donc, pour `θ > 0` : deux équilibres purs + un mixte = multiplicité.** Le jeu seul ne dit pas lequel se réalisera.

In [2]:
# === Section 1 : le jeu de coordination complet (information complète) ===

def equilibres_purs(theta):
    """Liste des équilibres de Nash purs du jeu 2x2 de coordination."""
    ne = []
    if theta > 0:            # (H,H) : H meilleure réponse a H  <=>  theta > 0
        ne.append(("H", "H"))
    ne.append(("L", "L"))    # (L,L) : toujours un équilibre (0 > -C)
    return ne

def equilibre_mixte(theta):
    """Proba p* d'indifférence (mixte) ; None si hors de [0, 1]."""
    # Indifférence :  p* (theta) + (1 - p*) (-C) = 0   =>   p* = C / (theta + C)
    p = C / (theta + C)
    return p if 0.0 <= p <= 1.0 else None

for theta in (0.5, -0.5):
    print(f"\ntheta = {theta:+.1f}")
    print("  Nash purs   :", equilibres_purs(theta))
    pm = equilibre_mixte(theta)
    print("  Nash mixte p* :", (round(pm, 4) if pm is not None else "aucun (hors [0,1])"))
print("\n-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.")


theta = +0.5
  Nash purs   : [('H', 'H'), ('L', 'L')]
  Nash mixte p* : 0.6667

theta = -0.5
  Nash purs   : [('L', 'L')]
  Nash mixte p* : aucun (hors [0,1])

-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.


## 2. Le signal bruité et la stratégie à seuil

Personne ne connaît `θ` directement. Chaque joueur `i` observe un **signal bruité**

$$s_i = \theta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \ \text{(indépendants).}$$

On restreint aux **stratégies à seuil** : jouer **H** si et seulement si `s_i ≥ k`. La décision se résume à une **frontière** `k` dans l'espace des signaux — la classe de stratégies naturelle quand l'information est un signal du fondamental.

In [3]:
# === Section 2 : le signal bruité + stratégie à seuil ===

def signaux(theta, sigma, n=1, rng=None):
    """n signaux bruités d'un même fondamental : s = theta + N(0, sigma^2)."""
    rng = rng or np.random.default_rng()
    return theta + rng.normal(0.0, sigma, size=n)

theta0, sigma0 = 0.5, 0.3
ex = signaux(theta0, sigma0, n=6, rng=np.random.default_rng(42))
print(f"theta = {theta0},  sigma = {sigma0}")
print("6 signaux tirés :", np.round(ex, 3))
print("-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.")

theta = 0.5,  sigma = 0.3
6 signaux tirés : [ 0.591  0.188  0.725  0.782 -0.085  0.109]
-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.


## 3. L'équilibre symétrique **unique** — et pourquoi le bruit tranche

Avec un **a priori** `θ ~ N(0, τ²)` et un signal gaussien, la croyance a posteriori d'un joueur de signal `s` est gaussienne : `θ | s ~ N(m, v)` avec

$$m = \tfrac{v\,s}{\sigma^2}, \qquad v = \Big(\tfrac{1}{\tau^2} + \tfrac{1}{\sigma^2}\Big)^{-1}.$$

Face à un adversaire de seuil `k`, l'utilité espérée de **H** s'écrit **en forme close** (intégration gaussienne) :

$$\mathbb{E}[U_H \mid s] = (m+C)\,\Phi\!\Big(\tfrac{m-k}{w}\Big) + \tfrac{v}{w}\,\varphi\!\Big(\tfrac{m-k}{w}\Big) - C, \qquad w = \sqrt{v + \sigma^2},$$

où `Φ` et `φ` sont la fonction de répartition et la densité de la normale standard. Un **seuil d'équilibre** `k*` est une valeur où un joueur de signal `s = k*` est **indifférent** face à un adversaire de même seuil : c'est une racine de `g(k) = U_H(k; k) = 0`. Le **plus petit** de ces seuils est l'équilibre **robuste** (les seuils élevés, s'ils existent, sont instables).

**Le point clé** : le plus petit seuil d'équilibre est **unique et bien défini**, et, quand le bruit disparaît, il converge vers le **coût d'agir seul** : `k* → C`. Le bruit remplace donc la multiplicité (deux purs + un mixte) par **un seuil unique**.

In [4]:
# === Section 3 : l'équilibre de seuil symétrique UNIQUE ===
# A priori theta ~ N(0, TAU2).  Signal : s = theta + eps, eps ~ N(0, sigma^2).

def post(s, sigma):
    """Croyance a posteriori theta | s : (moyenne m, variance v)."""
    v = 1.0 / (1.0 / TAU2 + 1.0 / sigma**2)
    m = v * s / sigma**2
    return m, v

def utilite_H(s, k, sigma):
    """Utilité espérée de H pour un joueur de signal s, face a un seuil adverse k."""
    m, v = post(s, sigma)
    w = math.sqrt(v + sigma**2)
    z = (m - k) / w
    return (m + C) * Phi(z) + (v / w) * phi(z) - C

def g_sym(k, sigma):
    """Indifférence symétrique : utilite_H d'un joueur de signal s = k face a un
    adversaire de MÊME seuil k. Les racines de g_sym sont les équilibres de seuil."""
    return utilite_H(k, k, sigma)

def seuil_equilibre(sigma, k_max=30.0, n=2000):
    """Le plus PETIT seuil d'équilibre (le robuste) : première racine de g_sym,
    trouvée par balayage depuis k=0 puis bisection."""
    g0 = g_sym(0.0, sigma)
    if g0 >= 0.0:
        return 0.0
    ks = np.linspace(0.0, k_max, n)
    for i in range(n - 1):
        a, b = ks[i], ks[i + 1]
        ga = g_sym(a, sigma)
        if ga * g_sym(b, sigma) <= 0.0:          # racine dans [a, b]
            for _ in range(80):
                mid = 0.5 * (a + b)
                if ga * g_sym(mid, sigma) <= 0.0:
                    b = mid
                else:
                    a, ga = mid, g_sym(mid, sigma)
            return 0.5 * (a + b)
    return float("nan")

def compte_racines(sigma, k_max=30.0, n=4000):
    """Nombre de seuils d'équilibre (changements de signe de g_sym sur [0, k_max])."""
    ks = np.linspace(0.0, k_max, n)
    g = [g_sym(k, sigma) for k in ks]
    return int(sum(1 for i in range(n - 1) if g[i] * g[i + 1] < 0))

sigma = 0.3
kstar = seuil_equilibre(sigma)
print(f"sigma = {sigma}   ->   plus petit seuil d'équilibre (robuste)  k* = {kstar:.4f}")

sigma = 0.3   ->   plus petit seuil d'équilibre (robuste)  k* = 1.5908


In [5]:
# Unicité : le nombre de seuils d'équilibre s'effondre à UN SEUL quand sigma diminue.
print("le bruit fait disparaître la multiplicité de SEUILS :")
print(f"{'sigma':>7}  {'nb de seuils k*':>16}  {'seuil robuste k*':>18}")
for sigma in (0.3, 0.2, 0.1, 0.05, 0.01, 0.001):
    print(f"{sigma:>7.3f}  {compte_racines(sigma):>16}  {seuil_equilibre(sigma):>18.4f}")

le bruit fait disparaître la multiplicité de SEUILS :
  sigma   nb de seuils k*    seuil robuste k*
  0.300                 2              1.5908
  0.200                 2              1.2334
  0.100                 2              1.0799
  0.050                 1              1.0335
  0.010                 1              1.0058
  0.001                 1              1.0006


In [6]:
# Probabilité (interne) de jouer H à l'équilibre : la part de joueurs qui jouent H.
#   s ~ N(theta, sigma^2)   =>   P(H) = P(s >= k*) = Phi((theta - k*) / sigma).
theta, sigma = 1.2, 0.3
kstar = seuil_equilibre(sigma)
p_H = Phi((theta - kstar) / sigma)
print(f"theta = {theta},  sigma = {sigma},  k* = {kstar:.4f}")
print(f"P(jouer H à l'équilibre) = {p_H:.4f}   ->  interne : UN seuil unique fixe UNE part de population (pas une multiplicité).")

theta = 1.2,  sigma = 0.3,  k* = 1.5908


P(jouer H à l'équilibre) = 0.0964   ->  interne : UN seuil unique fixe UNE part de population (pas une multiplicité).


### La sélection robuste (le résultat)

Au seuil unique, la part de joueurs qui jouent H est `P(H) = Φ((θ − k*)/σ)` — une valeur **interne** (ni 0 ni 1) : un seul équilibre, pas une multiplicité.

**Quand le bruit disparaît** (`σ → 0`), `k* → C` et les signaux convergent vers `θ`, donc :

- si **`θ > C`** : `P(les deux jouent H) → 1`  →  **l'équilibre EFFICACE (H,H) est sélectionné** ;
- si **`θ < C`** : `P(les deux jouent H) → 0`  →  l'équilibre (L,L) est sélectionné — et c'est un **échec de coordination** dès que `0 < θ < C` (l'équilibre efficace (H,H) existait, le bruit ne le sélectionne pas).

**Le bruit, au lieu d'embrouiller, fait le tri** autour du seuil `C` = coût d'agir seul. C'est le théorème central des jeux globaux.

In [7]:
# === La sélection robuste : sigma -> 0  =>  (H,H) si theta > C, sinon (L,L) ===
#   P(les deux jouent H) = [ Phi((theta - k*(sigma)) / sigma) ]^2   (signaux indépendants).

def prob_both_H(theta, sigma):
    k = seuil_equilibre(sigma)
    p = Phi((theta - k) / sigma)
    return p * p

sigmas = [0.3, 0.2, 0.1, 0.05, 0.01, 0.001]

print("theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(1.2, s):>18.4f}")

print("\ntheta = +0.8   (theta < C=1  ->  échec de coordination : (L,L) sélectionné, bien que (H,H) soit efficace)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(0.8, s):>18.4f}")

theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)


  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0093
  0.200     1.2334              0.1880
  0.100     1.0799              0.7836
  0.050     1.0335              0.9991
  0.010     1.0058              1.0000
  0.001     1.0006              1.0000

theta = +0.8   (theta < C=1  ->  échec de coordination : (L,L) sélectionné, bien que (H,H) soit efficace)
  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0000
  0.200     1.2334              0.0002
  0.100     1.0799              0.0000
  0.050     1.0335              0.0000
  0.010     1.0058              0.0000
  0.001     1.0006              0.0000


## 4. Synthèse

| Information | Équilibres | Résultat quand `σ → 0` |
|---|---|---|
| **Complète** (`θ` connu) | **multiplicité** : (H,H), (L,L), mixte | — |
| **Incomplète** (signal `s = θ + ε`) | **unique** : un seuil `k*` | **(H,H) si `θ > C`**, sinon (L,L) |

Le bruit ne dégrade pas le jeu : il **le résout** — au contraire de l'intuition. C'est ce qui rend les jeux globaux un outil central pour les crises de change, les paniques bancaires et la coordination sur un fondamental.

## 5. Exercices

Trois exercices pour consolider. Chaque fonction est un **stub** : complétez le corps. Les cellules s'exécutent telles quelles (elles renvoient `None` tant qu'incomplètes) — aucune erreur volontaire.

In [8]:
# Exercice 1 — La sélection robuste, côté L
def prob_both_L(theta, sigmas):
    """Pour chaque sigma, proba ex-post que les DEUX jouent L (cas theta < 0).

    Attendu : une liste de même longueur que sigmas ; quand sigma -> 0 et theta < 0,
    chaque valeur doit tendre vers 1.
    """
    # TODO : compléter
    # Indice : symétrique de prob_both_H. P(un joueur joue L) = 1 - Phi((theta - k*)/sigma),
    #         avec k* = seuil_equilibre(sigma) ; les signaux sont indépendants.
    result = None  # TODO etudiant
    return result

print(prob_both_L(-0.5, [0.5, 0.1, 0.01]))   # -> None tant que non complété

None


In [9]:
# Exercice 2 — Le seuil se fixe sur le coût d'agir seul
def seuils_en_fonction_de_sigma(sigmas):
    """Renvoie k*(sigma) pour chaque sigma.

    Attendu : on doit voir k* -> C (= 1 ici) quand sigma -> 0 (le seuil robuste se fixe
    sur le coût d'agir seul, pas sur 0).
    """
    # TODO : compléter
    # Indice : compréhension de liste — [seuil_equilibre(s) for s in sigmas]
    result = None  # TODO etudiant
    return result

print(seuils_en_fonction_de_sigma([1.0, 0.3, 0.05, 0.001]))   # -> None tant que non complété

None


In [10]:
# Exercice 3 — L'unique, vraiment unique ?
def est_unique(sigma, k_max=30.0):
    """True si, pour ce sigma, il n'existe qu'UN seuil d'équilibre (g_sym a une seule
    racine dans [0, k_max]) : le test d'UNICITÉ.
    """
    # TODO : compléter
    # Indice : balayer g_sym sur [0, k_max] et compter les changements de signe ;
    #         renvoyer True s'il y en a exactement un.
    result = None  # TODO etudiant
    return result

print(est_unique(0.05))   # -> None tant que non complété

None


## Références

- **Kay, J. & Perry, M. (2006).** *Global Games.* In : Arrow, Kenen & Stiglitz (eds), *Handbook of Economics and Finance*, Elsevier.
- **Hagen, T. & Köbis, J. (2016).** *Global Games : A Brief Overview.* (note de synthèse / cours).
- Fil d'origine : **Morris (1994)**, *Trading on Private Information* — le modèle de base.

**Place dans la série** : `11c`, « Autour de 11 » — avec `11` (jeux bayésiens), `11b` (Lean), `11d` (raffinements séquentiels).